In [10]:
import data_generation_utils as dgu

instances = dgu.read_instances("instances.txt")
instances

[Instance(n_processors=2, tasks=[Task(id=0, r=4, l=8, w=5), Task(id=1, r=6, l=9, w=11), Task(id=2, r=5, l=10, w=10), Task(id=3, r=3, l=9, w=1), Task(id=4, r=4, l=10, w=3), Task(id=5, r=3, l=5, w=9), Task(id=6, r=5, l=1, w=9)])]

In [11]:
import numpy as np
from math import factorial
from itertools import permutations
from data_generation_utils import Instance, Task


def evaluate_schedule(order, n_processors):
    """
    Schedule tasks in the given order on identical processors.

    Each task is assigned to the processor that becomes available
    earliest. Release times are respected.

    Returns:
        weighted_completion_time
        schedule
    """

    # When each processor becomes available
    processor_available = [0] * n_processors

    # (task_id, processor, start, completion)
    schedule = []

    objective = 0

    for task in order:
        # Find processor that becomes available first
        processor = min(
            range(n_processors),
            key=lambda p: processor_available[p]
        )

        start = max(
            processor_available[processor],
            task.r
        )

        completion = start + task.l

        processor_available[processor] = completion

        objective += task.w * completion

        schedule.append(
            (task.id, processor, start, completion)
        )

    return objective, schedule


def brute_force(instance: Instance):
    """
    Find the optimal schedule by enumerating all task permutations.

    WARNING:
        Complexity is O(n! * n * m), so this is only practical
        for small instances.
    """

    best_objective = float("inf")
    best_order = None
    best_schedule = None

    i = 0
    from math import factorial
    total = factorial(len(instance.tasks))
    for order in permutations(instance.tasks):

        objective, schedule = evaluate_schedule(
            order,
            instance.n_processors
        )

        if objective < best_objective:
            best_objective = objective
            best_order = order
            best_schedule = schedule
            
                # Print every 1% of progress
        if i % max(1, total // 100) == 0 or i == total:
            progress = i / total * 100

            print(
                f"\rProgress: {progress:6.2f}% "
                f"({i:,}/{total:,}) "
                f"Best objective: {best_objective:,}",
                end=""
            )
        i+=1

    return best_objective, best_order, best_schedule


    


In [12]:
result = brute_force(instances[0])
result

Progress:  99.21% (5,000/5,040) Best objective: 702

(702,
 (Task(id=5, r=3, l=5, w=9),
  Task(id=6, r=5, l=1, w=9),
  Task(id=1, r=6, l=9, w=11),
  Task(id=2, r=5, l=10, w=10),
  Task(id=0, r=4, l=8, w=5),
  Task(id=4, r=4, l=10, w=3),
  Task(id=3, r=3, l=9, w=1)),
 [(5, 0, 3, 8),
  (6, 1, 5, 6),
  (1, 1, 6, 15),
  (2, 0, 8, 18),
  (0, 1, 15, 23),
  (4, 0, 18, 28),
  (3, 1, 23, 32)])

In [13]:
import csv
import numpy as np
import time

header = ["dimension", "average_objective", "time"]

with open("brute_force", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for k in range(4, 12):
        instances = dgu.read_instances(f"data/{k}_tasks.txt")
        start = time.perf_counter()

        results = [brute_force(instance) for instance in instances]
        objectives = [result[0] for result in results]
        average = np.average(objectives)

        end = time.perf_counter()
        duration = end - start
        writer.writerow([k, average, duration])
        
    for k in range(12, 26):
        writer.writerow([k, None, None])

Progress:  22.36% (161/720) Best objective: 165

Progress:  77.38% (3,900/5,040) Best objective: 249

Progress:  87.30% (4,400/5,040) Best objective: 120

Progress:  91.27% (4,600/5,040) Best objective: 145

Progress:  48.61% (2,450/5,040) Best objective: 173

...

Progress:  72.42% (3,650/5,040) Best objective: 175

Progress:  44.64% (2,250/5,040) Best objective: 183

Progress:  82.34% (4,150/5,040) Best objective: 178

Progress:  97.22% (4,900/5,040) Best objective: 99

Progress:  99.00% (39,517,632/39,916,800) Best objective: 76684